# Sisyphus extraction pipeline

Two stages, composed with the `+` operator:

```
Stage 1 — Label    Filter(source_db) + Labeling(...) + Saver('labeled_db')
Stage 2 — Extract  Filter(labeled_db) + load + Extraction(...) + Writer(result_db)
                                                [+ optional Merger(fn)]
```

Prereq: the source `DocDB` already exists (run the indexing step once per
article set; see `sisyphus/index/`).

The three reference templates in `agent/references/` are complete files
you can copy as starting points — the cells below mirror them at a glance.

## 1. Single-property pipeline (bandgap)

**Stage 1 — Label** with one regex labeler, save to a labeled DB.

In [ ]:
import re
from sisyphus.chain import Filter, Labeler, Labeling, Saver, run_chains_with_extarction_history_multi_threads
from sisyphus.utils.helper_functions import get_plain_articledb

bandgap_labeler = Labeler(
    'band_gap',
    regex=re.compile(r'\b(band[- ]?gaps?|bandgaps?|energy[- ]?gap|energy gap)\b', re.I),
)

stage1_chain = (
    Filter(get_plain_articledb('nlo'))
    + Labeling(bandgap_labeler)
    + Saver('nlo_labeled')
)

# Single-file smoke test, then bulk:
# stage1_chain.compose('10.1002&sol;adfm.201801589.html')
run_chains_with_extarction_history_multi_threads(
    stage1_chain,
    'test_file',
    10,
    'nlo_labeled',
)

**Stage 2 — Extract**. Define a Pydantic schema, use `list[Bandgap]` directly (the framework auto-wraps it), and a small `Extractor` subclass.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from typing import Literal, Optional

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from sisyphus.chain import Extraction, Extractor, Filter, Paragraph, Writer
from sisyphus.utils.helper_functions import get_create_resultdb, get_plain_articledb


class Bandgap(BaseModel):
    """One bandgap measurement."""
    bandgap: Optional[str] = Field(description="Value with unit, e.g. '1.5 eV'")
    bandgap_type: Optional[Literal['direct', 'indirect']] = Field(description="Type")
    measurement_method: Optional[str] = Field(description="e.g. 'UV-Vis spectroscopy'")


PROMPT = ChatPromptTemplate([
    ('system', 'You extract information from scientific papers.'),
    ('user', '[START OF PAPER]\n{text}\n[END OF PAPER]\n\nInstruction:\n{instruction}'),
])


class BandgapExtractor(Extractor):
    properties = ['band_gap']
    schema = list[Bandgap]                   # framework auto-wraps list[X]
    model = ChatOpenAI(model_name='gpt-4.1', temperature=0)
    prompt = PROMPT
    strategy = 'merged'                       # one rich call per paper

    def build_prompt_vars(self, paragraph):
        return {'instruction': 'Extract bandgap value, type, and measurement method.'}


def load_from_labeled_db(docs):
    return [Paragraph.from_labeled_document(doc, id_) for id_, doc in enumerate(docs)]


stage2_chain = (
    Filter(get_plain_articledb('nlo_labeled'))
    + load_from_labeled_db
    + Extraction(BandgapExtractor())
    + Writer(get_create_resultdb('nlo_results'))
)

stage2_chain.compose('10.1002&sol;adfm.201801589.html')  # single-file example

## 2. Multiple independent properties

Bundle several Labelers and several Extractors. They run in parallel.
See `agent/references/multi_props_isolated.py` for a complete file.

```python
stage1_chain = (
    Filter(source_db)
    + Labeling(strength_labeler, phase_labeler, grain_size_labeler)
    + Saver('heas_labeled')
)

stage2_chain = (
    Filter(labeled_db)
    + load_from_labeled_db
    + Extraction(StrengthExtractor(), PhaseExtractor(), GrainSizeExtractor())
    + Writer(result_db)
)
```

## 3. Coupled properties + synthesis context (HEAs)

When properties refer back to the synthesis section (typical HEAs case),
use `strategy = 'merged'` plus `context_properties = ['synthesis']` so the
extractor always sees synthesis paragraphs as context.

Labelers can mix `regex` + `semantic` + `llm` filters in any combination:

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import dspy

from sisyphus.chain import Labeler, SemanticConfig

lm = dspy.LM('openai/gpt-4.1', max_tokens=3000)
dspy.configure(lm=lm)
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')
chroma_db = Chroma(collection_name='synthesis_embedding', embedding_function=embeddings)

In [ ]:
import re
from typing import Literal
import dspy

from sisyphus.chain import Filter, Labeler, Labeling, Saver, SemanticConfig
from sisyphus.chain.paragraph import Paragraph
from sisyphus.utils.helper_functions import get_plain_articledb


# DSPy signatures used as LLM filters
class LabelTablesStrength(dspy.Signature):
    """Given a CSV table from a HEAs paper, decide whether it contains
    at least one tensile/compressive test property (YS, UTS, fracture strain).
    Exclude hardness, fatigue, shear strength."""
    table: str = dspy.InputField()
    contains: bool = dspy.OutputField()


class ClassifySyn(dspy.Signature):
    """Assign topic to a paragraph from a HEAs paper. A 'synthesis' paragraph
    must describe synthesis/processing (melting, casting, rolling, annealing).
    Be strict."""
    paragraph: str = dspy.InputField()
    topic: Literal['synthesis', 'characterization', 'others'] = dspy.OutputField()


_table_classifier = dspy.ChainOfThought(LabelTablesStrength)
_topic_classifier = dspy.ChainOfThought(ClassifySyn)


def is_strength_table(paragraph):
    if not paragraph.is_table:
        return False
    return _table_classifier(table=paragraph.page_content).contains


def is_synthesis(paragraph):
    return _topic_classifier(paragraph=paragraph.page_content).topic == 'synthesis'


# Labelers — declarative, no subclassing needed
strength_labeler = Labeler(
    'strength',
    regex=re.compile(r'(\b(MPa|GPa)\b|\d+(\.\d+)?\s*%)'),
    semantic=SemanticConfig(
        vector_store=chroma_db,
        query=(
            "The stress-strain curve of alloy, describes yield strength (ys), "
            "tensile strength (uts) and elongation properties."
        ),
        section_pattern=re.compile(r'result', re.I),
        k=5,
    ),
)

strength_table_labeler = Labeler('strength', llm=is_strength_table)

phase_labeler = Labeler(
    'phase',
    regex=re.compile(
        r'\b(FCC|BCC|HCP|L12|B2|Laves|face-centered cubic|body-centered cubic|hexagonal close-packed|intermetallic)\b',
        re.I,
    ),
    semantic=SemanticConfig(
        vector_store=chroma_db,
        query="Microstructure characterization of alloys (FCC, BCC, HCP, L12, B2 ...)",
        section_pattern=re.compile(r'result', re.I),
        k=5,
    ),
)

experimental_labeler = Labeler(
    'synthesis',
    semantic=SemanticConfig(
        vector_store=chroma_db,
        query="Experimental procedures for synthesis and processing of HEAs.",
        section_pattern=re.compile(r'(experiment)|(preparation)|(method)', re.I),
        k=3,
    ),
    llm=is_synthesis,
)


stage1_chain = (
    Filter(get_plain_articledb('heas_1531'))
    + Labeling(strength_labeler, strength_table_labeler, phase_labeler, experimental_labeler)
    + Saver('heas_labeled')
)

stage1_chain.compose('10.1002&sol;adem.201600726.html')

### Stage 2 — HEAs extraction with dynamic schema

An Extractor with `strategy = 'merged'` concatenates target paragraphs +
anything in `context_properties` (here: synthesis) into one rich-context call.

Override `build_schema(paragraph)` / `build_prompt_vars(paragraph)` to
adapt per paragraph — the HEAs extractor below only includes fields whose
labels are actually present on the merged paragraph, so the LLM doesn't
waste tokens on irrelevant fields.

The full HEAs Pydantic models, instruction blocks, and DSPy template
selector are in `agent/references/multi_props.py`. The skeleton is:

```python
from sisyphus.chain import Extractor

class HeaExtractor(Extractor):
    properties         = ['strength', 'phase', 'grain_size', 'synthesis']
    context_properties = ['synthesis']     # always carry synthesis as context
    strategy           = 'merged'
    model              = ChatOpenAI(model_name='gpt-4.1', temperature=0)
    prompt             = PROMPT

    def build_schema(self, paragraph):
        props_present = [p for p in ['strength', 'phase', 'grain_size'] if paragraph.has(p)]
        return _build_records_model(props_present, paragraph.is_synthesis)

    def build_prompt_vars(self, paragraph):
        return {'instruction': _build_instruction(paragraph)}
```

## 4. Optional post-extraction Merger

For per-paper dedup or entity resolution, slot a `Merger` between Extraction
and Writer. It receives `list[Extracted]` and returns `list[Extracted]`:

```python
from sisyphus.chain import Merger

def dedupe_by_composition(items):
    ...
    return items

chain = (
    Filter(labeled_db)
    + load_from_labeled_db
    + Extraction(...)
    + Merger(dedupe_by_composition)
    + Writer(result_db)
)
```

## Where to go from here

- **`sisyphus/chain/SKILL.md`** — the single-page reference for the API.
- **`agent/references/single_prop.py`** — minimal single-property template.
- **`agent/references/multi_props_isolated.py`** — multi-property template.
- **`agent/references/multi_props.py`** — coupled HEAs-style template.